# NBA production run — pinned warehouse

This notebook is orchestration only. It runs the standalone `nba-backend` pipeline against **wyattowalsh/basketball version 238**, writes the normalized cache outside the repository, and prepares only `nba-backend/data_delivery` for human-reviewed publication. It never commits or pushes.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

DATASET = "wyattowalsh/basketball"
DATASET_VERSION = "238"
EXPECTED_REPO = "andrewkemmer/sports_prediction_model"
REPO = Path("/kaggle/working/sports_prediction_model")
DELIVERY = Path("nba-backend/data_delivery")

os.environ["NBA_KAGGLE_DATASET_VERSION"] = DATASET_VERSION
os.environ["NBA_PUSH"] = "0"
# Keep the derived warehouse outside the checkout. Kaggle's working
# directory is ephemeral, but this also makes the scope explicit.
os.environ.setdefault("NBA_CACHE_DIR", "/kaggle/working/nba-cache")

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{EXPECTED_REPO}.git", str(REPO)], check=True)
assert (REPO / "nba-backend/backend/master_pipeline.py").is_file()
print(f"cloned {EXPECTED_REPO} at NBA dataset pin {DATASET_VERSION}")

In [ ]:
subprocess.run(["pip", "install", "-q", "-r", "nba-backend/backend/requirements.txt"], cwd=REPO, check=True)
print("NBA dependencies installed")

In [ ]:
# Resolve the attached Kaggle input when the dataset is mounted by the
# notebook UI, otherwise download the exact pinned version. The pipeline
# accepts the extracted directory or its nba.duckdb/nba.sqlite file.
def find_warehouse():
    candidates = [Path("/kaggle/input/basketball"), Path("/kaggle/input/wyattowalsh-basketball")]
    candidates.extend(sorted(Path("/kaggle/input").glob("**/nba.duckdb")))
    candidates.extend(sorted(Path("/kaggle/input").glob("**/nba.sqlite")))
    for candidate in candidates:
        is_warehouse_dir = candidate.is_dir() and any(
            (candidate / marker).exists()
            for marker in ("nba.duckdb", "nba.sqlite", "dim_game.csv",
                            "dim_game.parquet", "parquet", "csv")
        )
        if is_warehouse_dir or (candidate.is_file() and candidate.name in {"nba.duckdb", "nba.sqlite"}):
            return candidate
    return None

source = find_warehouse()
if source is None:
    target = Path("/kaggle/working/nba-warehouse")
    if target.exists():
        shutil.rmtree(target)
    subprocess.run([
        "kaggle", "datasets", "download", "-d", DATASET,
        "-v", DATASET_VERSION, "--unzip", "-p", str(target),
    ], check=True)
    source = target
    if not source.exists():
        raise RuntimeError("Pinned Kaggle download did not create a warehouse directory")
print(f"warehouse source: {source}")

In [ ]:
cmd = ["python", "nba-backend/backend/master_pipeline.py", "--source-path", str(source), "--skip-pull"]
result = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
if result.returncode != 0:
    raise SystemExit(f"NBA pipeline failed with exit code {result.returncode}")
print("NBA production pipeline completed")

In [ ]:
# Delivery boundary audit: stage ONLY the NBA delivery directory. This is
# intentionally not a commit and not a push; publication remains a
# separate human-reviewed action.
staged = subprocess.run(
    ["git", "diff", "--name-only", "--", str(DELIVERY)],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout.splitlines()
outside = [p for p in staged if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside:
    raise RuntimeError(f"delivery scope violation: {outside}")
subprocess.run(["git", "add", "--", str(DELIVERY)], cwd=REPO, check=True)
cached = subprocess.run(
    ["git", "diff", "--cached", "--name-only"], cwd=REPO,
    capture_output=True, text=True, check=True,
).stdout.splitlines()
outside_cached = [p for p in cached if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside_cached:
    raise RuntimeError(f"staged scope violation: {outside_cached}")
print(f"staged {len(cached)} NBA delivery files; no commit or push performed")